# Notebook 01 — Data Ingestion
## NYC Taxi Trips — Big Data Pipeline

**Archivos esperados en `data/raw/`:**
- `yellow_tripdata_2023-01.parquet`
- `yellow_tripdata_2023-02.parquet`
- `yellow_tripdata_2023-03.parquet`
- `NYC_Taxi_Zones.csv`

In [1]:
# ─── Celda 1: Imports y configuración de rutas ───────────────────────────────
import os
import json
import datetime
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Rutas del proyecto — ajusta BASE_PATH si tu carpeta está en otro lugar
BASE_PATH    = os.path.abspath("../")          # raíz del proyecto
RAW_PATH     = os.path.join(BASE_PATH, "data", "raw")
LOG_PATH     = os.path.join(BASE_PATH, "data", "raw", "ingestion_log.json")
SCREENS_PATH = os.path.join(BASE_PATH, "screenshots")

os.makedirs(SCREENS_PATH, exist_ok=True)

print("BASE_PATH :", BASE_PATH)
print("RAW_PATH  :", RAW_PATH)
print("Archivos en raw:")
for f in sorted(os.listdir(RAW_PATH)):
    size_mb = os.path.getsize(os.path.join(RAW_PATH, f)) / (1024 * 1024)
    print(f"  {f:<45} {size_mb:>8.1f} MB")

BASE_PATH : c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project
RAW_PATH  : c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\raw
Archivos en raw:
  NYC_Taxi_Zones.csv                                 3.6 MB
  ingestion_log.json                                 0.0 MB
  yellow_tripdata_2023-01.parquet                   45.5 MB
  yellow_tripdata_2023-02.parquet                   45.5 MB
  yellow_tripdata_2023-03.parquet                   53.5 MB


In [2]:
# ─── Celda 2: Iniciar SparkSession ───────────────────────────────────────────
import os
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_Ingestion") \
    .master("local[2]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Spark OK ✅")

Spark version: 3.5.1
Spark OK ✅


In [3]:
# ─── Celda 3: Cargar los 3 archivos Parquet de trips ─────────────────────────
import pandas as pd

parquet_files = [
    os.path.join(RAW_PATH, "yellow_tripdata_2023-01.parquet"),
    os.path.join(RAW_PATH, "yellow_tripdata_2023-02.parquet"),
    os.path.join(RAW_PATH, "yellow_tripdata_2023-03.parquet"),
]

pdf = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)

# Convertir fechas a string para que Spark las acepte sin problema
pdf["tpep_pickup_datetime"]  = pdf["tpep_pickup_datetime"].astype(str)
pdf["tpep_dropoff_datetime"] = pdf["tpep_dropoff_datetime"].astype(str)

total_rows = len(pdf)
total_cols = len(pdf.columns)

print(f"Total de filas   : {total_rows:,}")
print(f"Total de columnas: {total_cols}")

df_trips = spark.createDataFrame(pdf)
print("✅ DataFrame Spark creado correctamente")

Total de filas   : 9,384,487
Total de columnas: 20
✅ DataFrame Spark creado correctamente


In [4]:
# ─── Celda 4: Vista previa ────────────────────────────────────────────────────
print("Primeras 5 filas:")
print(pdf.head(5).to_string())

print("\nEstadísticas básicas:")
print(pdf[["trip_distance","fare_amount","tip_amount","total_amount","passenger_count"]].describe())

Primeras 5 filas:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  total_amount  congestion_surcharge  airport_fee  Airport_fee
0         2  2023-01-01 00:32:10   2023-01-01 00:40:36              1.0           0.97         1.0                  N           161           141             2          9.3   1.00      0.5        0.00           0.0                    1.0         14.30                   2.5         0.00          NaN
1         2  2023-01-01 00:55:08   2023-01-01 01:01:27              1.0           1.10         1.0                  N            43           237             1          7.9   1.00      0.5        4.00           0.0                    1.0         16.90                   2.5         0.00          NaN
2         2  2023-01-01 00:25:04   2023-01-01 00:37:49              1.0           

In [5]:
# ─── Celda 5: Cargar Taxi Zones CSV ──────────────────────────────────────────
zones_path = os.path.join(RAW_PATH, "NYC_Taxi_Zones.csv")

df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(zones_path)

zones_rows = df_zones.count()
print(f"Taxi Zones — filas: {zones_rows}")
print("Columnas:", df_zones.columns)
df_zones.show(5)

Taxi Zones — filas: 263
Columnas: ['Shape Geometry', 'Shape Length', 'Shape Area', 'Zone', 'Location ID', 'Borough']
+--------------------+---------------+----------------+--------------------+-----------+-------------+
|      Shape Geometry|   Shape Length|      Shape Area|                Zone|Location ID|      Borough|
+--------------------+---------------+----------------+--------------------+-----------+-------------+
|MULTIPOLYGON (((-...| 0.116357453189|  7.823067885E-4|      Newark Airport|          1|          EWR|
|MULTIPOLYGON (((-...|  0.43346966679|0.00486634037837|         Jamaica Bay|          2|       Queens|
|MULTIPOLYGON (((-...|0.0843411059012|3.14414156821E-4|Allerton/Pelham G...|          3|        Bronx|
|MULTIPOLYGON (((-...|0.0435665270921|1.11871946192E-4|       Alphabet City|          4|    Manhattan|
|MULTIPOLYGON (((-...|0.0921464898574|4.97957489363E-4|       Arden Heights|          5|Staten Island|
+--------------------+---------------+----------------+----

In [6]:
# ─── Celda 6: Validación de calidad básica ────────────────────────────────────
print("=== Conteo de valores nulos por columna ===")
print(pdf.isnull().sum())

print("\n=== Rango de fechas en el dataset ===")
print("Fecha mínima:", pdf["tpep_pickup_datetime"].min())
print("Fecha máxima:", pdf["tpep_pickup_datetime"].max())

print("\n=== Distribución por mes ===")
pdf["mes"] = pd.to_datetime(pdf["tpep_pickup_datetime"]).dt.month
print(pdf["mes"].value_counts().sort_index())

=== Conteo de valores nulos por columna ===
VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count           236179
trip_distance                  0
RatecodeID                236179
store_and_fwd_flag        236179
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge      236179
airport_fee              6389464
Airport_fee              3231202
dtype: int64

=== Rango de fechas en el dataset ===
Fecha mínima: 2001-01-01 00:06:49
Fecha máxima: 2023-04-05 20:17:42

=== Distribución por mes ===
mes
1     3066733
2     2914003
3     3403619
4          85
10         11
11          1
12         35
Name: count, dtype: int64


In [7]:
# ─── Celda 7: Generar log de ingesta ─────────────────────────────────────────
ingestion_log = {
    "timestamp": datetime.datetime.now().isoformat(),
    "project": "NYC Taxi Big Data Pipeline",
    "sources": [
        {
            "file": "yellow_tripdata_2023-01.parquet",
            "type": "parquet",
            "size_mb": round(os.path.getsize(parquet_files[0]) / (1024*1024), 2)
        },
        {
            "file": "yellow_tripdata_2023-02.parquet",
            "type": "parquet",
            "size_mb": round(os.path.getsize(parquet_files[1]) / (1024*1024), 2)
        },
        {
            "file": "yellow_tripdata_2023-03.parquet",
            "type": "parquet",
            "size_mb": round(os.path.getsize(parquet_files[2]) / (1024*1024), 2)
        },
        {
            "file": "NYC_Taxi_Zones.csv",
            "type": "csv",
            "size_mb": round(os.path.getsize(zones_path) / (1024*1024), 2)
        }
    ],
    "total_trip_records": total_rows,
    "total_zone_records": zones_rows,
    "columns": df_trips.columns,
    "status": "SUCCESS"
}

with open(LOG_PATH, "w") as f:
    json.dump(ingestion_log, f, indent=2)

print("Log de ingesta guardado en:", LOG_PATH)
print(json.dumps(ingestion_log, indent=2))

Log de ingesta guardado en: c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\data\raw\ingestion_log.json
{
  "timestamp": "2026-04-14T04:01:03.471569",
  "project": "NYC Taxi Big Data Pipeline",
  "sources": [
    {
      "file": "yellow_tripdata_2023-01.parquet",
      "type": "parquet",
      "size_mb": 45.46
    },
    {
      "file": "yellow_tripdata_2023-02.parquet",
      "type": "parquet",
      "size_mb": 45.54
    },
    {
      "file": "yellow_tripdata_2023-03.parquet",
      "type": "parquet",
      "size_mb": 53.53
    },
    {
      "file": "NYC_Taxi_Zones.csv",
      "type": "csv",
      "size_mb": 3.58
    }
  ],
  "total_trip_records": 9384487,
  "total_zone_records": 263,
  "columns": [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_

In [8]:
# ─── Celda 8: Guardar evidencia ───────────────────────────────────────────────
sample_path = os.path.join(SCREENS_PATH, "sample_trips_ingested.csv")
pdf.head(100).to_csv(sample_path, index=False)
print(f"Muestra guardada en: {sample_path}")

zones_sample_path = os.path.join(SCREENS_PATH, "sample_zones_ingested.csv")
df_zones_pd = pd.read_csv(zones_path)
df_zones_pd.to_csv(zones_sample_path, index=False)
print(f"Zonas guardadas en : {zones_sample_path}")

print("\n✅ Notebook 01 completado exitosamente.")
print(f"   Total registros ingestados: {total_rows:,}")
print(f"   Zonas de taxi cargadas    : {len(df_zones_pd)}")

Muestra guardada en: c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\screenshots\sample_trips_ingested.csv
Zonas guardadas en : c:\Users\IV4M\Desktop\IA\8vo\DatosMasivos\nyc_taxi_project\screenshots\sample_zones_ingested.csv

✅ Notebook 01 completado exitosamente.
   Total registros ingestados: 9,384,487
   Zonas de taxi cargadas    : 263


In [9]:
# ─── Celda 9: Cerrar Spark ────────────────────────────────────────────────────
spark.stop()
print("SparkSession cerrada.")

SparkSession cerrada.
